# Stage 2 — Inference Acceleration
## Notebook 1: Quantization Techniques

**Objective:** Understand and apply quantization to compress LLMs for efficient inference — from FP32 down to INT4 — using AutoGPTQ, AutoAWQ, GGUF, and KV-cache quantization.

**Prerequisites:** PyTorch fundamentals, familiarity with Hugging Face Transformers.

In [ ]:
# ============================
# Cell 1: Install dependencies
# ============================
!pip install -q torch transformers accelerate auto-gptq autoawq optimum bitsandbytes huggingface_hub
!pip install -q sentencepiece tiktoken protobuf
!pip install -q matplotlib seaborn psutil datasets

## 1. Why Quantization?

LLMs are huge. A 7B-parameter model in FP32 requires ~28 GB just to load the weights. This makes deployment expensive and slow. **Quantization** reduces the bit-width of each weight, trading a small amount of accuracy for dramatic savings in memory and latency.

### The core idea

```
Original weight (FP32): 0.123456789 → 32 bits
     |
     v   drop 24 bits
Quantized weight (INT8): 0.1235      →  8 bits   (4x smaller)
```

**Key terms:**
- **Quantization** — mapping continuous values to a discrete set
- **Dequantization** — mapping discrete values back to continuous
- **Scale factor (s)** — the step size between quantized levels
- **Zero point (z)** — maps the real zero to its quantized representation

### The quantization formula

$$x_{\text{quant}} = \text{round}\left(\frac{x_{\text{float}}}{s}\right) + z$$

$$x_{\text{dequant}} = (x_{\text{quant}} - z) \times s$$

## 2. FP32 → FP16 → BF16 → INT8 → INT4: The Precision Spectrum

| Format | Bits per value | Range (approx) | Memory (7B model) | Typical accuracy |
|--------|---------------|----------------|-------------------|------------------|
| FP32 | 32 | +-3.4 x 10^38 | 28 GB | Baseline |
| FP16 | 16 | +-65,504 | 14 GB | Matches FP32 |
| BF16 | 16 | +-3.4 x 10^38 | 14 GB | Matches FP32 |
| INT8 | 8 | -128 to 127 | 7 GB | ~99% of FP16 |
| INT4 | 4 | -8 to 7 | 3.5 GB | ~95-98% of FP16 |

### Why does BF16 exist alongside FP16?

FP16 has a small dynamic range (max 65,504) so gradients overflow easily during training. BF16 keeps the same 8-bit exponent as FP32 but with fewer mantissa bits — same range, less precision. For inference, FP16 is fine; for training, BF16 is safer.

```
FP32: [ sign (1) | exponent (8)  | mantissa (23) ]
BF16: [ sign (1) | exponent (8)  | mantissa (7)  ]  <- same range as FP32
FP16: [ sign (1) | exponent (5)  | mantissa (10) ]  <- smaller range, more precision
```

## 3. Symmetric vs Asymmetric Quantization

### Symmetric Quantization

Zero-point `z = 0`. The quantized range is symmetric about zero.

```
Float:   -max  .....  0  .....  +max
            |            |            |
INT8:     -127  .....  0  .....  +127
                      z=0
```

**Pros:** Simpler math (no zero-point), hardware-friendly.  
**Cons:** Wastes half the range if values are all-positive (e.g., ReLU outputs).

### Asymmetric Quantization

Zero-point `z != 0`. The quantized range is shifted to match the data distribution.

```
Float:    min  .....  0  .......  max
            |            |            |
INT8:       0  .....  z  .......  255
```

**Pros:** Full range utilization, better for skewed distributions (typical in activations).  
**Cons:** Extra zero-point term in dot-product math, slightly slower.

### When to use which

- **Weights** -> symmetric (weights are roughly symmetric around zero)
- **Activations** -> asymmetric (ReLU/GELU outputs are non-negative)

In [ ]:
# ================================================
# Cell 4: Demonstrate symmetric vs asymmetric quant
# ================================================
import numpy as np
import matplotlib.pyplot as plt

def symmetric_quantize(tensor, n_bits=8):
    """Symmetric quantization: z=0, scale = max(|x|) / (2^(n_bits-1) - 1)"""
    qmax = 2 ** (n_bits - 1) - 1
    scale = np.max(np.abs(tensor)) / qmax
    q = np.clip(np.round(tensor / scale), -qmax, qmax).astype(np.int8)
    dq = q.astype(np.float32) * scale
    return q, scale, 0, dq

def asymmetric_quantize(tensor, n_bits=8):
    """Asymmetric quantization: scale = (max-min)/(2^n_bits-1), z = -min/scale"""
    qmin, qmax = 0, 2 ** n_bits - 1
    tmin, tmax = np.min(tensor), np.max(tensor)
    scale = (tmax - tmin) / (qmax - qmin)
    zero_point = qmin - np.round(tmin / scale)
    zero_point = np.clip(zero_point, qmin, qmax)
    q = np.clip(np.round(tensor / scale) + zero_point, qmin, qmax).astype(np.uint8)
    dq = (q.astype(np.float32) - zero_point) * scale
    return q, scale, zero_point, dq

# Create a skewed distribution (like activations after ReLU)
np.random.seed(42)
data = np.random.randn(1000).astype(np.float32) * 0.5 + 0.8
data = np.clip(data, -2, 3)

q_sym, s_sym, z_sym, dq_sym = symmetric_quantize(data, 8)
q_asym, s_asym, z_asym, dq_asym = asymmetric_quantize(data, 8)

mse_sym  = np.mean((data - dq_sym) ** 2)
mse_asym = np.mean((data - dq_asym) ** 2)

print(f"Symmetric   - scale={s_sym:.4f},  z={z_sym},   MSE={mse_sym:.6f}")
print(f"Asymmetric  - scale={s_asym:.4f}, z={z_asym},  MSE={mse_asym:.6f}")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist(data, bins=50, alpha=0.7, label='Original (FP32)')
axes[0].set_title('Original Data Distribution')
axes[1].hist(dq_sym, bins=50, alpha=0.7, color='orange', label='Symmetric INT8')
axes[1].set_title(f'Symmetric (MSE={mse_sym:.4f})')
axes[2].hist(dq_asym, bins=50, alpha=0.7, color='green', label='Asymmetric INT8')
axes[2].set_title(f'Asymmetric (MSE={mse_asym:.4f})')
plt.tight_layout()
plt.show()

## 4. Quantization Granularity

How do we group weights for computing scale/zero-point?

| Granularity | Scale computed per... | Accuracy | Speed |
|-------------|----------------------|----------|-------|
| **Per-tensor** | Entire weight tensor | Lowest | Fastest |
| **Per-channel** | Each output channel | High | Moderate |
| **Per-group** | Group of N elements (e.g., 128) | Highest | Slower |

Most modern quantization methods (GPTQ, AWQ) use **per-group** quantization with groups of 128 elements for the best accuracy-speed tradeoff.

In [ ]:
# =======================================
# Cell 6: Per-tensor vs per-channel demo
# =======================================
import torch

# Simulate a weight matrix: shape [out_features, in_features]
W = torch.randn(512, 256) * 0.3  # channels have different variances
W[0:100] *= 3.0   # channels 0-99 have larger range
W[100:200] *= 0.5 # channels 100-199 have smaller range

def per_tensor_quant(w):
    scale = w.abs().max() / 127
    return (w / scale).round().clamp(-128, 127).to(torch.int8), scale

def per_channel_quant(w):
    scales = w.abs().max(dim=1).values / 127  # shape [out_features]
    q = (w / scales.unsqueeze(1)).round().clamp(-128, 127).to(torch.int8)
    return q, scales

q_tensor, s_tensor = per_tensor_quant(W)
q_channel, s_channel = per_channel_quant(W)

# Dequantize and compute error per channel
dq_tensor = q_tensor.float() * s_tensor
dq_channel = q_channel.float() * s_channel.unsqueeze(1)

ch_errors_tensor = ((W - dq_tensor) ** 2).mean(dim=1)
ch_errors_channel = ((W - dq_channel) ** 2).mean(dim=1)

print(f"Per-tensor  MSE (avg across channels): {ch_errors_tensor.mean():.6f}")
print(f"              Max channel error:          {ch_errors_tensor.max():.6f}")
print()
print(f"Per-channel MSE (avg across channels): {ch_errors_channel.mean():.6f}")
print(f"              Max channel error:          {ch_errors_channel.max():.6f}")
print()
print(f"Improvement factor: {ch_errors_tensor.mean() / ch_errors_channel.mean():.1f}x")

## 5. Comparison Table: Precision / Size / Speed / Accuracy

Below is a comprehensive comparison of quantization levels for LLM inference on a 7B model.

In [ ]:
# ==============================================
# Cell 8: Comprehensive comparison table
# ==============================================
import pandas as pd

comparison_data = {
    "Precision": ["FP32", "FP16", "BF16", "INT8 (dynamic)", "INT4 (GPTQ)", "INT4 (AWQ)", "NF4 (QLoRA)", "GGUF Q4_K_M"],
    "Bits per weight": [32, 16, 16, 8, 4, 4, 4, 4.34],
    "Model Size (7B)": ["28 GB", "14 GB", "14 GB", "7 GB", "3.9 GB", "3.9 GB", "3.9 GB", "4.1 GB"],
    "Speed (tok/s)*": ["45", "80", "80", "120", "180", "195", "110", "25 (CPU)"],
    "Memory Reduction": ["1x", "2x", "2x", "4x", "~7.2x", "~7.2x", "~7.2x", "~6.8x"],
    "Quality Loss": ["None", "Negligible", "Negligible", "< 0.2%", "1-3%", "0.5-1.5%", "2-5%", "1-2%"],
    "Inference Engine": ["Transformers", "vLLM/TGI", "vLLM/TGI", "vLLM/TGI", "vLLM/ExLlama", "vLLM", "bitsandbytes", "llama.cpp"],
    "Pre-quantize Needed?": ["No", "No", "No", "No", "Yes (slow)", "Yes (faster)", "No (on-load)", "Yes"],
}

df = pd.DataFrame(comparison_data)
print("LLM Quantization Comprehensive Comparison")
print("=" * 100)
print(df.to_string(index=False))
print()
print("*Speed measured on A100 GPU unless noted as CPU")

## 6. GPTQ — Post-Training Quantization

**GPTQ** (Frantar et al., 2023) applies layer-wise quantization using second-order information (Hessian) to minimize output error. It quantizes weights column-by-column, adjusting remaining columns to compensate.

**Key properties:**
- Quantizes each layer independently (no retraining needed)
- Needs a small calibration dataset (~128 samples)
- Produces INT4 weights with group-wise scaling
- Inference kernels fuse dequantization into the matmul

**Algorithm sketch:**
```
For each layer:
  1. Compute inverse Hessian H^-1 from calibration data
  2. For each column j of the weight matrix:
     a. Quantize column j -> w_q[j]
     b. Compute quantization error e = w[j] - w_q[j]
     c. Update remaining columns: W[:, j+1:] -= (H^-1[:,j] / H^-1[j,j]) * e
```

In [ ]:
# ========================================
# Cell 10: AutoGPTQ demo on Qwen2.5-0.5B
# ========================================
# NOTE: This cell requires a GPU with sufficient VRAM.
# For CPU-only environments, skip to the GGUF section.

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import time
import os

MODEL_ID = "Qwen/Qwen2.5-0.5B"

# Helper: measure VRAM
def get_gpu_memory():
    if torch.cuda.is_available():
        return torch.cuda.memory_allocated() / 1024**3
    return 0.0

# 1. Load FP16 baseline
print("=" * 60)
print("Loading FP16 model...")
t0 = time.time()
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model_fp16 = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto" if torch.cuda.is_available() else "cpu",
    trust_remote_code=True
)
load_time_fp16 = time.time() - t0
vram_fp16 = get_gpu_memory()

param_count = sum(p.numel() for p in model_fp16.parameters())
size_gb_fp16 = param_count * 2 / 1024**3  # FP16 = 2 bytes per param
print(f"Parameters: {param_count/1e9:.2f}B")
print(f"Model size (FP16): {size_gb_fp16:.2f} GB")
print(f"VRAM allocated:   {vram_fp16:.2f} GB")
print(f"Load time:        {load_time_fp16:.1f}s")

# Inference helper
def run_inference(model, tok, prompt, max_tokens=30):
    inputs = tok(prompt, return_tensors="pt")
    if torch.cuda.is_available():
        inputs = {k: v.cuda() for k, v in inputs.items()}
    with torch.no_grad():
        t_start = time.time()
        outputs = model.generate(**inputs, max_new_tokens=max_tokens, do_sample=False)
        dt = time.time() - t_start
    return tok.decode(outputs[0], skip_special_tokens=True), dt

prompt = "Explain what machine learning is in one sentence:"
result_fp16, time_fp16 = run_inference(model_fp16, tokenizer, prompt)
print(f"\nFP16 Output: {result_fp16}")
print(f"FP16 Inference time: {time_fp16:.2f}s")

In [ ]:
# ========================================
# Cell 11: Quantize with AutoGPTQ (INT4)
# ========================================
# NOTE: GPTQ requires a GPU for both quantization and inference.

print("Preparing GPTQ quantization...")

try:
    from auto_gptq import AutoGPTQForCausalLM, BaseQuantizeConfig
    from datasets import load_dataset

    quantize_config = BaseQuantizeConfig(
        bits=4,
        group_size=128,
        desc_act=False,
        damp_percent=0.01,
        sym=True,
    )

    # Load calibration data (~128 samples)
    calib_data = load_dataset("wikitext", "wikitext-2-raw-v1", split="train")
    calib_texts = [sample["text"] for sample in calib_data.select(range(128)) if len(sample["text"]) > 50]
    print(f"Calibration samples: {len(calib_texts)}")

    t_quant_start = time.time()
    print("Quantizing with GPTQ... (this takes a few minutes)")

    model_gptq = AutoGPTQForCausalLM.from_pretrained(
        MODEL_ID,
        quantize_config=quantize_config,
        trust_remote_code=True
    )
    tokenizer.pad_token = tokenizer.eos_token
    calib_tokens = tokenizer(calib_texts, truncation=True, max_length=512, padding=False)
    model_gptq.model.resize_token_embeddings(len(tokenizer))
    model_gptq.quantize(calib_tokens, batch_size=1)

    t_quant = time.time() - t_quant_start
    vram_gptq = get_gpu_memory()

    save_dir = "/tmp/qwen_gptq_int4"
    model_gptq.save_quantized(save_dir, use_safetensors=True)
    size_gptq_gb = param_count * 0.55 / 1024**3

    print(f"\nGPTQ-INT4 Quantization done in {t_quant:.1f}s")
    print(f"Estimated model size: {size_gptq_gb:.2f} GB")
    print(f"VRAM allocated:      {vram_gptq:.2f} GB")
    print(f"Compression ratio:   {size_gb_fp16 / size_gptq_gb:.1f}x smaller than FP16")

    # Inference test
    result_gptq, time_gptq = run_inference(model_gptq.model, tokenizer, prompt, max_tokens=30)
    print(f"\nGPTQ-INT4 Output: {result_gptq}")
    print(f"GPTQ-INT4 Inference time: {time_gptq:.2f}s")

except ImportError as e:
    print(f"AutoGPTQ not available: {e}")
    print("Install with: pip install auto-gptq")
    model_gptq = None
    vram_gptq = 0
    size_gptq_gb = 0
    time_gptq = 0
except Exception as e:
    print(f"GPTQ quantization failed (likely no GPU): {e}")
    model_gptq = None
    vram_gptq = 0
    size_gptq_gb = 0
    time_gptq = 0

## 7. AWQ — Activation-Aware Weight Quantization

**AWQ** (Lin et al., 2024) observes that not all weight channels are equally important — channels corresponding to large activations matter more. It protects these salient channels by scaling them up before quantization and scaling down after.

**Key insight:** A small fraction (~1%) of weight channels have outsized importance. By finding these channels and protecting them, AWQ avoids the quality degradation that naive rounding INT4 quantization would cause.

**AWQ pipeline:**
1. Run a few calibration samples through the FP16 model
2. For each linear layer, measure which input channels have the largest activations
3. Scale up the corresponding weight channels by a per-channel factor `s`
4. Quantize the scaled weights to INT4
5. At runtime, the scaling is folded into the preceding layer norm

**GPTQ vs AWQ comparison:**

| Aspect | GPTQ | AWQ |
|--------|------|-----|
| Core idea | 2nd-order error correction | Salient channel protection |
| Calibration data | 128 samples | 128 samples |
| Quantization time | Moderate | Faster |
| Accuracy (INT4) | Good | Slightly better |
| Kernel support | ExLlama, Marlin | AWQ kernels |

In [ ]:
# ========================================
# Cell 13: AutoAWQ demo on Qwen2.5-0.5B
# ========================================
# NOTE: AWQ requires a GPU.

print("Quantizing with AWQ...")

try:
    from awq import AutoAWQForCausalLM

    t_awq_start = time.time()

    model_awq = AutoAWQForCausalLM.from_pretrained(
        MODEL_ID,
        trust_remote_code=True,
        safetensors=True
    )

    quant_config_awq = {
        "zero_point": True,
        "q_group_size": 128,
        "w_bit": 4,
        "version": "GEMM",
    }

    model_awq.quantize(
        tokenizer,
        quant_config=quant_config_awq,
        calib_data=[calib_texts[i] for i in range(min(64, len(calib_texts)))]
    )

    t_awq_quant = time.time() - t_awq_start
    vram_awq = get_gpu_memory()
    size_awq_gb = param_count * 0.55 / 1024**3

    print(f"AWQ-INT4 Quantization done in {t_awq_quant:.1f}s")
    print(f"Estimated model size: {size_awq_gb:.2f} GB")
    print(f"VRAM allocated:      {vram_awq:.2f} GB")

    # Save
    save_dir = "/tmp/qwen_awq_int4"
    model_awq.save_quantized(save_dir)
    tokenizer.save_pretrained(save_dir)

    # Inference test
    result_awq, time_awq = run_inference(model_awq.model, tokenizer, prompt, max_tokens=30)
    print(f"\nAWQ-INT4 Output: {result_awq}")
    print(f"AWQ-INT4 Inference time: {time_awq:.2f}s")

except ImportError as e:
    print(f"AutoAWQ not available: {e}")
    model_awq = None
    vram_awq = 0
    size_awq_gb = 0
    time_awq = 0
except Exception as e:
    print(f"AWQ quantization failed (likely no GPU): {e}")
    model_awq = None
    vram_awq = 0
    size_awq_gb = 0
    time_awq = 0

In [ ]:
# ==========================================
# Cell 14: Benchmark — FP16 vs GPTQ vs AWQ
# ==========================================
print("=" * 70)
print("BENCHMARK SUMMARY: FP16 vs GPTQ-INT4 vs AWQ-INT4")
print("=" * 70)

print(f"\n{'Method':<15} {'Time (s)':<12} {'VRAM (GB)':<12} {'Size (GB)':<12}")
print(f"{'-'*60}")
if 'time_fp16' in dir():
    print(f"{'FP16':<15} {time_fp16:<12.3f} {vram_fp16:<12.2f} {size_gb_fp16:<12.2f}")
if model_gptq is not None:
    print(f"{'GPTQ-INT4':<15} {time_gptq:<12.3f} {vram_gptq:<12.2f} {size_gptq_gb:<12.2f}")
if model_awq is not None:
    print(f"{'AWQ-INT4':<15} {time_awq:<12.3f} {vram_awq:<12.2f} {size_awq_gb:<12.2f}")
print(f"{'='*70}")

# Test multiple prompts
test_prompts = [
    "What is quantum computing?",
    "Write a haiku about programming.",
    "Explain the theory of relativity in simple terms.",
]

results = []

print("\nPer-prompt benchmarks:")
for prompt_text in test_prompts:
    print(f"\n  Prompt: {prompt_text[:60]}...")
    out_fp16, t_fp16_p = run_inference(model_fp16, tokenizer, prompt_text, max_tokens=50)
    n_tok = len(tokenizer.encode(out_fp16)) - len(tokenizer.encode(prompt_text))
    print(f"    FP16:     {n_tok} tokens in {t_fp16_p:.2f}s ({n_tok/t_fp16_p:.1f} tok/s)")
    if model_gptq is not None:
        out_gptq, t_gptq_p = run_inference(model_gptq.model, tokenizer, prompt_text, max_tokens=50)
        print(f"    GPTQ-INT4: {n_tok} tokens in {t_gptq_p:.2f}s ({n_tok/t_gptq_p:.1f} tok/s)")
    if model_awq is not None:
        out_awq, t_awq_p = run_inference(model_awq.model, tokenizer, prompt_text, max_tokens=50)
        print(f"    AWQ-INT4:  {n_tok} tokens in {t_awq_p:.2f}s ({n_tok/t_awq_p:.1f} tok/s)")

# Cleanup
del model_fp16
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## 8. GGUF and llama.cpp — CPU-Friendly Quantization

**GGUF** (GGML Universal Format) is the model format used by `llama.cpp`. It bundles the model weights, tokenizer, and metadata into a single file and supports dozens of quantization types optimized for CPU inference.

### Common GGUF quantization types

| Type | Bits/weight | Quality | Speed | Use case |
|------|------------|---------|-------|----------|
| **Q2_K** | 2.56 | Lowest | Fastest | Extreme compression |
| **Q3_K_M** | 3.35 | Low | Fast | Memory-constrained |
| **Q4_K_M** | 4.34 | Medium | Fast | **Best balanced choice** |
| **Q5_K_M** | 5.45 | High | Moderate | Quality-focused |
| **Q6_K** | 6.56 | Higher | Moderate | Near-lossless |
| **Q8_0** | 8.00 | Highest | Slower | Lossless (for reference) |
| **F16** | 16.00 | Reference | Slowest | Baseline |

### What do K-quants mean?

- **K** = uses importance-based mixed precision (important layers get more bits)
- **_S** = Small (less VRAM, slightly lower quality)
- **_M** = Medium (balanced)
- **_L** = Large (best quality, more VRAM)

### Convert a model to GGUF

```bash
# 1. Convert HF model to FP16 GGUF
python convert_hf_to_gguf.py /path/to/model --outtype f16 --outfile model-f16.gguf

# 2. Quantize to Q4_K_M
./llama-quantize model-f16.gguf model-Q4_K_M.gguf Q4_K_M
```

In [ ]:
# =============================================
# Cell 16: GGUF quantization level reference
# =============================================
print("Common llama.cpp quantization presets:")
print("=" * 70)

quant_types = [
    ("Q2_K", 2.56, "Extreme compression, significant quality loss"),
    ("Q3_K_S", 3.25, "Small variant, low quality"),
    ("Q3_K_M", 3.45, "Medium variant, usable for simple tasks"),
    ("Q3_K_L", 3.55, "Large variant, best Q3 quality"),
    ("Q4_0", 4.50, "Legacy 4-bit, fast on CPU"),
    ("Q4_K_S", 4.78, "Small 4-bit K-quant"),
    ("Q4_K_M", 4.85, "Best practical trade-off (RECOMMENDED)"),
    ("Q5_0", 5.00, "Legacy 5-bit"),
    ("Q5_K_S", 5.45, "Small 5-bit K-quant"),
    ("Q5_K_M", 5.75, "High quality, noticeable improvement"),
    ("Q6_K", 6.56, "Near-perfect, diminishing returns"),
    ("Q8_0", 8.50, "Virtually lossless"),
    ("F16", 16.0, "Full precision, identical to source"),
]

print(f"{'Type':<10s} {'Bits/Weight':>12s}  {'Description'}")
print(f"{'-'*70}")
for name, bits, desc in quant_types:
    print(f"{name:<10s} {bits:>10.2f}    {desc}")

print()
print("Quick rule: For any model, start with Q4_K_M.")
print("If quality is insufficient, go to Q5_K_M.")
print("If you need to fit on edge devices, go to Q3_K_M.")

## 9. KV Cache Quantization

The KV cache stores keys and values for past tokens. For long sequences, the KV cache can consume more memory than the model weights themselves. Quantizing the KV cache saves memory with minimal quality loss.

In [ ]:
# =============================================
# Cell 18: KV Cache Quantization — memory measurement
# =============================================

def simulate_kv_cache(
    batch_size: int,
    num_layers: int,
    num_heads: int,
    head_dim: int,
    sequence_length: int,
    kv_cache_dtype: str = "fp16"
):
    """Calculate KV cache memory usage for given parameters."""
    bytes_per_element = {
        "fp32": 4,
        "fp16": 2,
        "int8": 1,
        "int4": 0.5,
    }
    bpe = bytes_per_element[kv_cache_dtype]

    # KV cache = 2 (K + V) * batch_size * num_layers * num_heads * seq_len * head_dim
    total_elements = 2 * batch_size * num_layers * num_heads * sequence_length * head_dim
    memory_mb = total_elements * bpe / (1024 ** 2)

    return memory_mb


# Simulate a Llama-2-7B style model: 32 layers, 32 heads, head_dim=128
configs = [
    ("FP16 KV Cache", "fp16"),
    ("INT8 KV Cache", "int8"),
    ("INT4 KV Cache", "int4"),
]

seq_lengths = [512, 1024, 2048, 4096, 8192]

print("KV Cache Memory Usage (Llama-2-7B: 32L, 32H, 128D, BS=1)")
print(f"{'Seq Len':<12}", end="")
for name, _ in configs:
    print(f"{name:<20}", end="")
print(f"{'Savings (INT4 vs FP16)':<25}")
print("-" * 77)

for sl in seq_lengths:
    print(f"{sl:<12}", end="")
    mems = []
    for name, dtype in configs:
        mem = simulate_kv_cache(1, 32, 32, 128, sl, dtype)
        mems.append(mem)
        print(f"{mem:<20.0f} MB", end="")
    savings = (1 - mems[2] / mems[0]) * 100
    print(f"{savings:<25.1f}%")

print()
print("Key insight: For long sequences, INT4 KV cache quantization")
print("can save ~75% of KV cache memory with negligible accuracy loss.")

In [ ]:
# ============================================
# Cell 19: Visualize KV cache growth by method
# ============================================
import numpy as np

seq_lengths_arr = np.array([512, 1024, 2048, 4096, 8192, 16384, 32768, 65536, 131072])
cache_fp16 = np.array([simulate_kv_cache(1, 32, 32, 128, sl, "fp16") for sl in seq_lengths_arr])
cache_int8 = np.array([simulate_kv_cache(1, 32, 32, 128, sl, "int8") for sl in seq_lengths_arr])
cache_int4 = np.array([simulate_kv_cache(1, 32, 32, 128, sl, "int4") for sl in seq_lengths_arr])

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(seq_lengths_arr, cache_fp16, 'o-', label='FP16 KV Cache', linewidth=2)
ax.plot(seq_lengths_arr, cache_int8, 's-', label='INT8 KV Cache', linewidth=2)
ax.plot(seq_lengths_arr, cache_int4, '^-', label='INT4 KV Cache', linewidth=2)
ax.axhline(y=14000, color='gray', linestyle='--', alpha=0.7, label='Model weights (7B FP16) ~14 GB')
ax.fill_between(seq_lengths_arr, cache_fp16, cache_int8, alpha=0.15, label='FP16->INT8 savings')
ax.set_xlabel('Sequence Length (tokens)')
ax.set_ylabel('KV Cache Size (MB)')
ax.set_title('KV Cache Memory Growth for Llama-2-7B-Style Model')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xscale('log', base=2)
plt.tight_layout()
plt.show()

print("At 32K tokens, FP16 KV cache can exceed model weights!")
print("INT8 cuts that in half. INT4 makes it barely noticeable.")

## 10. PagedAttention — vLLM's Memory Magic

**PagedAttention** (Kwon et al., 2023) is the key innovation behind vLLM's high throughput. It applies OS virtual-memory paging to the KV cache.

### The problem with naive KV caching

```
Traditional approach: pre-allocate one big contiguous KV cache per sequence.
+----------------------------------------------------+
| Seq 1 KV cache (contiguous, max_len=2048)             |  <- 50% wasted if seq is 1024 tokens
+----------------------------------------------------+
| Seq 2 KV cache (contiguous, max_len=2048)             |  <- same waste
+----------------------------------------------------+
Memory fragmentation + external fragmentation.
```

### The PagedAttention solution

```
PagedAttention: break KV cache into fixed-size blocks (like memory pages).

+----+----+----+----+----+----+----+----+
| S1 | S2 | S1 | S1 | S2 | -  | S3 | S2 |  <- blocks shared across sequences
+----+----+----+----+----+----+----+----+
                                  ^free

Benefits:
1. No pre-allocation - allocate pages on demand
2. No fragmentation - blocks are uniform
3. Memory sharing - prefix caching across sequences
4. Higher throughput - less memory waste = more sequences in parallel
```

### Memory savings example

```
Without PagedAttention: 4 sequences x 2048 max tokens = 8GB reserved
With PagedAttention:    4 sequences x avg 500 tokens = 2GB actual + free list
                        -----------------------------
                        6 GB saved -> more throughput!
```

In [ ]:
# ============================================
# Cell 21: PagedAttention simulation
# ============================================
from dataclasses import dataclass, field
from typing import List, Dict

@dataclass
class KVCacheBlock:
    """A single block of KV cache (like a page)."""
    block_id: int
    token_capacity: int = 16
    ref_count: int = 0

@dataclass
class PageTable:
    """Maps logical token positions to physical blocks."""
    blocks: List[int] = field(default_factory=list)

class PagedKVCache:
    """A simple PagedAttention simulator."""
    def __init__(self, num_blocks: int, block_size: int = 16):
        self.block_size = block_size
        self.free_blocks = {i for i in range(num_blocks)}
        self.used_blocks: Dict[int, KVCacheBlock] = {}
        self.page_tables: Dict[int, PageTable] = {}

    def allocate(self, seq_id: int, num_tokens: int) -> bool:
        """Allocate pages for a sequence."""
        num_pages = (num_tokens + self.block_size - 1) // self.block_size
        if len(self.free_blocks) < num_pages:
            return False  # OOM

        pt = PageTable()
        for _ in range(num_pages):
            bid = self.free_blocks.pop()
            self.used_blocks[bid] = KVCacheBlock(bid, self.block_size, 1)
            pt.blocks.append(bid)
        self.page_tables[seq_id] = pt
        return True

    def share_prefix(self, seq_id: int, source_seq_id: int, prefix_tokens: int):
        """Share prefix blocks between sequences."""
        num_pages = (prefix_tokens + self.block_size - 1) // self.block_size
        source_pt = self.page_tables[source_seq_id]
        pt = PageTable(blocks=source_pt.blocks[:num_pages].copy())
        for bid in pt.blocks:
            self.used_blocks[bid].ref_count += 1
        self.page_tables[seq_id] = pt

    def free(self, seq_id: int):
        """Free pages for a sequence."""
        if seq_id not in self.page_tables:
            return
        for block_id in self.page_tables[seq_id].blocks:
            block = self.used_blocks.get(block_id)
            if block:
                block.ref_count -= 1
                if block.ref_count <= 0:
                    del self.used_blocks[block_id]
                    self.free_blocks.add(block_id)
        del self.page_tables[seq_id]

    @property
    def memory_usage(self):
        return len(self.used_blocks) * self.block_size

    @property
    def stats(self):
        return {
            "used_blocks": len(self.used_blocks),
            "free_blocks": len(self.free_blocks),
            "active_requests": len(self.page_tables),
            "utilization_pct": len(self.used_blocks) / (len(self.used_blocks) + len(self.free_blocks)) * 100
        }

# Demo
cache = PagedKVCache(num_blocks=100, block_size=16)
print("=== PagedAttention Simulation ===\n")
print(f"Initial: {cache.stats}")

cache.allocate(1, 200)
print(f"After Seq1 (200 tokens): {cache.stats}")

cache.share_prefix(2, 1, 128)
cache.allocate(2, 128)
print(f"After Seq2 sharing 128-token prefix: {cache.stats}")
print(f"  Without sharing: 128*2={256} tokens; actual: {cache.memory_usage} tokens")

cache.allocate(3, 500)
print(f"After Seq3 (500 tokens): {cache.stats}")

cache.free(1)
print(f"After freeing Seq1: {cache.stats}")
print(f"  Note: shared blocks kept (ref_count > 0)")

cache.free(2)
print(f"After freeing Seq2: {cache.stats}")

print("\n=== Static vs Paged Comparison ===")
print(f"Static would reserve: 3 x max(200,128,500) = 1500 token-slots")
print(f"Paged peak usage: {cache.memory_usage} tokens at peak")

## 11. BitsAndBytes — On-the-Fly Quantization

`bitsandbytes` provides 4-bit and 8-bit quantization during model loading — no need to pre-quantize. Great for experimentation and QLoRA fine-tuning.

**NF4** (NormalFloat4) is a data-dependent quantization format: it assumes weights follow a normal distribution and spaces quantization levels accordingly — more levels near zero, fewer at extremes.

In [ ]:
# ============================================
# Cell 23: BitsAndBytes 4-bit loading demo
# ============================================
# NOTE: Requires bitsandbytes + GPU

print("Loading with bitsandbytes 4-bit...")

try:
    from transformers import BitsAndBytesConfig

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
        bnb_4bit_use_double_quant=True,
    )

    t0 = time.time()
    model_bnb = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto" if torch.cuda.is_available() else {"": "cpu"},
        trust_remote_code=True
    )
    load_time_bnb = time.time() - t0
    vram_bnb = get_gpu_memory()

    print(f"Loaded in {load_time_bnb:.1f}s")
    print(f"VRAM used: {vram_bnb:.2f} GB")

    out_bnb, t_bnb = run_inference(model_bnb, tokenizer, prompt, max_tokens=30)
    print(f"\nBNB-4bit Output: {out_bnb}")
    print(f"BNB-4bit Inference time: {t_bnb:.2f}s")

    del model_bnb
except Exception as e:
    print(f"bitsandbytes loading failed (likely no GPU): {e}")
    print("Skipping — this is expected on CPU-only machines.")

## 12. Quantization Decision Flowchart

```
Need quantization for:
|
+- CPU-only deployment? -> GGUF (llama.cpp) with Q4_K_M
|
+- GPU serving, need max throughput? -> GPTQ or AWQ + vLLM
|   +- NVIDIA GPU? -> GPTQ + Marlin kernel (fastest)
|
+- Just experimenting, want quick 4-bit? -> BitsAndBytes (no pre-quantize)
|
+- Fine-tuning on quantized model? -> QLoRA (BitsAndBytes NF4)
|
+- Long-context inference? -> Add KV cache quantization (INT8/INT4)
+- Edge/mobile? -> GGUF Q3_K_M or Q4_0
```

In [ ]:
# ============================================
# Cell 25: Exercise — Write a benchmarking script
# ============================================
"""
EXERCISE: Complete the benchmarking script below.
Replace each '# TODO' with the correct code.
"""

import torch
import time
import psutil
from transformers import AutoModelForCausalLM, AutoTokenizer

# Model to benchmark
EXERCISE_MODEL = "Qwen/Qwen2.5-0.5B"

TEST_PROMPTS = [
    ("short", "What is 2+2?"),
    ("medium", "Explain how a transformer neural network works in detail."),
    ("long", "Write a detailed essay about the history of AI from 1950 to 2024."),
]

def benchmark_model(model_id, dtype, quant_method="none", **load_kwargs):
    """Benchmark a model with given configuration.
    
    Args:
        model_id: HuggingFace model ID
        dtype: torch dtype
        quant_method: "none", "bnb-int8", "bnb-int4"
    
    Returns: dict with benchmark results
    """
    results = {
        "model": model_id,
        "dtype": str(dtype),
        "quant_method": quant_method,
    }
    
    # Load model
    t0 = time.time()
    
    # TODO 1: Load the tokenizer
    tokenizer = ...  # YOUR CODE HERE
    
    # TODO 2: Load the model with given dtype and kwargs
    model = ...  # YOUR CODE HERE
    
    results["load_time_s"] = time.time() - t0
    
    # TODO 3: Calculate model size in GB
    param_count = ...  # YOUR CODE HERE
    results["param_count"] = param_count / 1e9
    results["model_size_gb"] = ...  # YOUR CODE HERE
    
    if torch.cuda.is_available():
        results["vram_allocated_gb"] = torch.cuda.memory_allocated() / 1024**3
    
    # Run inference on each prompt
    results["per_prompt"] = []
    for label, prompt in TEST_PROMPTS:
        inputs = tokenizer(prompt, return_tensors="pt")
        if torch.cuda.is_available():
            inputs = {k: v.cuda() for k, v in inputs.items()}
        
        with torch.no_grad():
            t_start = time.time()
            # TODO 4: Generate up to 50 new tokens
            outputs = ...  # YOUR CODE HERE
            t_gen = time.time() - t_start
        
        new_tokens = outputs.shape[1] - inputs["input_ids"].shape[1]
        tok_per_sec = new_tokens / t_gen if t_gen > 0 else 0
        
        results["per_prompt"].append({
            "label": label,
            "new_tokens": new_tokens,
            "time_s": t_gen,
            "tokens_per_second": tok_per_sec,
        })
    
    # Aggregate
    results["avg_tokens_per_second"] = sum(
        p["tokens_per_second"] for p in results["per_prompt"]
    ) / len(results["per_prompt"])
    
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    
    return results


# Run benchmarks
def run_exercise():
    all_results = []
    
    print("=" * 60)
    print("Running benchmarks...")
    print("=" * 60)
    
    try:
        # TODO 5: Call benchmark_model for FP32
        results_fp32 = benchmark_model(EXERCISE_MODEL, torch.float32, "none")
        all_results.append(results_fp32)
        print(f"\nFP32: {results_fp32['avg_tokens_per_second']:.1f} tok/s, "
              f"{results_fp32['model_size_gb']:.2f} GB")
    except Exception as e:
        print(f"FP32 benchmark failed: {e}")
    
    # Print summary
    print("\n" + "=" * 60)
    print("FINAL SUMMARY")
    print("=" * 60)
    for r in all_results:
        print(f"{r['quant_method']:>10s} | {r['avg_tokens_per_second']:>8.1f} tok/s | "
              f"{r['model_size_gb']:>6.2f} GB | {r['load_time_s']:>5.1f}s load")
    
    return all_results

print("Exercise script ready.")
print("\nTODO hints:")
print("1. AutoTokenizer.from_pretrained(model_id)")
print("2. AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=dtype, ...)")
print("3. sum(p.numel() for p in model.parameters()); multiply by dtype byte size")
print("4. model.generate(**inputs, max_new_tokens=50)")
print("5. See the function signature above")

# Uncomment to run:
# run_exercise()

## 13. Key Takeaways

1. **Quantization maps FP32 weights to lower-bit integers** using scale factors and zero points, reducing model size by 2-8x with minimal accuracy loss.

2. **Symmetric quantization** is simpler and works well for weights. **Asymmetric** handles skewed distributions (like ReLU activations) better.

3. **GPTQ** uses second-order error correction for column-by-column weight quantization — good for GPU serving.

4. **AWQ** protects salient weight channels by scaling them before quantization — slightly better accuracy than GPTQ with faster quantization.

5. **GGUF/llama.cpp** is the go-to for CPU inference — Q4_K_M offers the best size/quality tradeoff.

6. **BitsAndBytes** supports on-the-fly 4-bit and 8-bit quantization — ideal for experimentation and QLoRA fine-tuning.

7. **KV cache quantization** (INT8/INT4) saves up to 75% of KV cache memory for long-context inference.

8. **PagedAttention** applies OS paging concepts to KV cache — enabling higher throughput through better memory utilization.

9. **Choice of quantization method depends on your deployment target:** GPU serving (GPTQ/AWQ), CPU (GGUF), or experimentation (BitsAndBytes).

### What's Next

Now that you understand how to compress models, the next notebook covers **vLLM** — the serving framework that combines PagedAttention with optimized kernels for maximum throughput.